In [27]:
from transformers import AutoTokenizer, AutoModelForQuestionAnswering
MODEL_B = "distilbert-base-uncased"
tokenizer_b = AutoTokenizer.from_pretrained(MODEL_B)
model_b = AutoModelForQuestionAnswering.from_pretrained(MODEL_B)
MAX_LENGTH = 384
DOC_STRIDE = 96

Loading weights: 100%|██████████| 100/100 [00:00<00:00, 6791.52it/s]
[transformers] DistilBertForQuestionAnswering LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
qa_outputs.weight       | MISSING    | 
qa_outputs.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [28]:
def prepare_qa_features(examples):
    tokenized = tokenizer_b(
        [q.strip() for q in examples["question"]],
        examples["context"],
        truncation="only_second",
        max_length=MAX_LENGTH,
        stride=DOC_STRIDE,
        return_overflowing_tokens=True,
        return_offsets_mapping=True,
        padding="max_length",
    )
    sample_mapping = tokenized.pop("overflow_to_sample_mapping")
    offsets = tokenized.pop("offset_mapping")
    start_positions, end_positions = [], []
    for feature_index, feature_offsets in enumerate(offsets):
        input_ids = tokenized["input_ids"][feature_index]
        cls_index = input_ids.index(tokenizer_b.cls_token_id)
        sequence_ids = tokenized.sequence_ids(feature_index)
        sample_index = sample_mapping[feature_index]
        answer_start = examples["answer_start"][sample_index]
        answer_end = answer_start + len(examples["answer_text"][sample_index])
        context_start = 0
        while sequence_ids[context_start] != 1:
            context_start += 1
        context_end = len(sequence_ids) - 1
        while sequence_ids[context_end] != 1:
            context_end -= 1
        if (
            feature_offsets[context_start][0] > answer_start
            or feature_offsets[context_end][1] < answer_end
        ):
            start_positions.append(cls_index)
            end_positions.append(cls_index)
            continue
        token_start = context_start
        while feature_offsets[token_start][1] <= answer_start:
            token_start += 1
        token_end = context_end
        while feature_offsets[token_end][0] >= answer_end:
            token_end -= 1
        start_positions.append(token_start)
        end_positions.append(token_end)
    tokenized["start_positions"] = start_positions
    tokenized["end_positions"] = end_positions
    return tokenized

In [29]:
# TODO(student 2): create a small SQuAD-style technical-support dataset
# with trusted contexts from the supplied KB/docs and at least 30 QA pairs.
qa_sample_rows = [
    {
        "question": "Which port does the technical support API use by default?",
        "context": "The technical support API listens on port 8000 by default.",
        "answer_text": "8000",
        "answer_start": 42,
    },
    {
        "question": "What does an HTTP 503 response indicate?",
        "context": "A 503 response indicates that the service is temporarily unavailable.",
        "answer_text": "temporarily unavailable",
        "answer_start": 45,
    },
    {
        "question": "What should happen when database corruption is suspected?",
        "context": "If database corruption is suspected, stop automated recovery and escalate the incident to a human operator.",
        "answer_text": "escalate the incident to a human operator",
        "answer_start": 65,
    },
]

In [ ]:
from datasets import Dataset
from transformers import DataCollatorWithPadding
qa_dataset = Dataset.from_list(qa_sample_rows)

qa_splits = qa_dataset.train_test_split(test_size=0.2,seed=42)

qa_train_features = qa_splits["train"].map(
    prepare_qa_features,
    batched=True,
    remove_columns=qa_splits["train"].column_names,
)

qa_val_features = qa_splits["test"].map(
    prepare_qa_features,
    batched=True,
    remove_columns=qa_splits["test"].column_names,
)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer_b)

Map: 100%|██████████| 1/1 [00:00<00:00, 502.01 examples/s]


In [31]:
from transformers import TrainingArguments, Trainer
args_b = TrainingArguments(
    output_dir="models/qa_model",
    learning_rate=3e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=2,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    report_to="none",
    save_total_limit=1,
)
trainer_b = Trainer(
    model=model_b,
    args=args_b,
    train_dataset=qa_train_features,
    eval_dataset=qa_val_features,
    data_collator=data_collator,
)
trainer_b.train()
trainer_b.save_model("models/qa_model")
tokenizer_b.save_pretrained("models/qa_model")

/Users/emad/Tuwaiq/Multi_Model_Agent_Architecture/.venv/lib/python3.13/site-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss
1,No log,5.839768
2,No log,5.809308


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  6.43it/s]
/Users/emad/Tuwaiq/Multi_Model_Agent_Architecture/.venv/lib/python3.13/site-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  6.36it/s]


('models/qa_model/tokenizer_config.json', 'models/qa_model/tokenizer.json')